# ROC (Receiver Operating Characteristic) curve & AUC (Area Under the Curve)

_Machine Learning_

**See how a classifier trades off catches against false alarms.**

A classifier outputs a probability. A threshold turns it into yes/no.

     Move the threshold and the catches and false alarms change together.

---

This notebook is a practice scaffold for the **ROC (Receiver Operating Characteristic) curve & AUC (Area Under the Curve)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Train a classifier and get probability scores

We make a synthetic two-class problem, split off a test set, and fit a `LogisticRegression`. The key step is `predict_proba(...)[:, 1]`: instead of a hard yes/no, we pull out each test example's **probability of class 1**. The ROC curve lives entirely in these continuous scores, not in hard predictions.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=500, n_features=6, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

clf = LogisticRegression().fit(Xtr, ytr)
scores = clf.predict_proba(Xte)[:, 1]   # probability of class 1

print("first 5 scores:", np.round(scores[:5], 3))

### Step 2 — Summarise the whole curve with AUC

**AUC** (area under the ROC curve) collapses performance across *every* threshold into one number: 0.5 means random guessing, 1.0 means perfect ranking. Concretely it's the probability that a random positive example scores higher than a random negative one.

In [ ]:
from sklearn.metrics import roc_auc_score

# One number summarising performance across all thresholds.
auc = roc_auc_score(yte, scores)
print("AUC:", round(auc, 4))   # 0.5 = random, 1.0 = perfect

### Step 3 — Walk the curve threshold by threshold

`roc_curve` returns the false-positive rate (FPR), true-positive rate (TPR), and the threshold at each operating point. Printing a few points shows the trade-off directly: a high threshold catches few positives but raises few false alarms, and lowering it raises both together.

In [ ]:
from sklearn.metrics import roc_curve

# Every operating point: false-positive rate, true-positive rate, threshold.
fpr, tpr, thresh = roc_curve(yte, scores)

# Peek at the first, middle, and last operating points.
points = [0, len(fpr) // 2, len(fpr) - 1]
for i in points:
    print("FPR %.2f  TPR %.2f  threshold %.2f" % (fpr[i], tpr[i], thresh[i]))

## Visualize the data & results

_Across every threshold, how well does the breast-cancer classifier trade true positives against false positives?_

### Step 1 — Train on the real breast-cancer data

We load the 569-patient breast-cancer dataset, **standardise** the features so logistic regression converges cleanly, and fit the classifier. As before we pull out the predicted probability of the positive (benign) class for the held-out test patients.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

bc = load_breast_cancer()
# Standardise the features so logistic regression converges cleanly.
X = StandardScaler().fit_transform(bc.data)
y = bc.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

clf = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
scores = clf.predict_proba(Xte)[:, 1]   # probability of benign

### Step 2 — Plot the ROC curve

`RocCurveDisplay.from_predictions` sweeps every threshold and draws TPR against FPR, annotating the AUC. The dashed diagonal is a random guesser (AUC 0.5); the further the curve bows toward the top-left corner, the better the classifier separates the classes.

In [ ]:
from sklearn.metrics import RocCurveDisplay

# Sweep all thresholds and draw TPR vs FPR, with the AUC annotated.
RocCurveDisplay.from_predictions(yte, scores)

# Dashed diagonal = a random guesser (AUC 0.5).
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.title("ROC curve (breast cancer)")
plt.show()